In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

port = pd.read_csv(OUT / "portfolio_C_overlap.csv")
rets = pd.read_csv(DATA_PROCESSED / "return_panel_500.csv", parse_dates=["date"])

port["ticker"] = port["ticker"].astype(str)
rets["ticker"] = rets["ticker"].astype(str)
port["report_date"] = pd.to_datetime(port["report_date"])
rets["date"] = pd.to_datetime(rets["date"])

pf = port.merge(rets, left_on=["report_date", "ticker"], right_on=["date", "ticker"], how="left")
pf["weighted_return"] = pf["weight"] * pf["return"]

bt = pf.groupby("report_date", as_index=False).agg(
    portfolio_return=("weighted_return", "sum"),
    n_names=("ticker", "nunique"),
    n_holdings=("selected", "sum")
)

bt["cum_return"] = (1 + bt["portfolio_return"].fillna(0)).cumprod() - 1
bt.to_csv(OUT / "backtest_C_overlap.csv", index=False)

print(bt.head())

  report_date  portfolio_return  n_names  n_holdings  cum_return
0  2020-07-31          0.056510       36          30    0.056510
1  2020-08-31          0.027151       14          14    0.085195
2  2020-09-30         -0.027834      260          30    0.054990
3  2020-10-31         -0.027953       36          30    0.025499
4  2020-11-30          0.068957       15          15    0.096215
